In [ ]:
import ee
import geemap #eemont
# ee.Authenticate()
# ee.Initialize()
from importlib import reload  

In [ ]:
geemap.ee_initialize(project="water-sinapohlabeln")

In [ ]:
from modules import high_level_functions_CO2
from modules import utils_Landsat_SR_CO2 as utils_LS
from modules import ms_indices_CO2 as indices
from modules import configs, utils_string

Reloading

In [ ]:
utils_string = reload(utils_string)
reload(high_level_functions_CO2)
reload(indices)
reload(utils_LS)

In [ ]:
# PROPERTIES
# SET METADATA PARAMETERS
MAXCLOUD = 70 #70
STARTYEAR = 2005
ENDYEAR = 2024
STARTMONTH = 7
ENDMONTH = 8
SCALE = 30
longitudes = [-151] #Besser: -155, #OG:-154 
latitudes = [59] #Besser: 70.5#OG:70
SIZE_LON = 10
SIZE_LAT = 2

#target_collection = 'users/ingmarnitze/TCTrend_SR_2005-2024_TCVIS'
#target_collection_nObs = 'users/ingmarnitze/TCTrend_SR_2004-2025_nObservations'

target_collection = 'users/ingmarnitze/TCTrend_SR_2005-2024_TCVIS'
target_collection_nObs = 'users/ingmarnitze/TCTrend_SR_2004-2025_nObservations'

In [ ]:
# image metadata Filters
config_trend = {
  'STARTYEAR': STARTYEAR,
  'ENDYEAR': ENDYEAR,
  'date_filter_yr' : ee.Filter.calendarRange(STARTYEAR, ENDYEAR, 'year'),
  'date_filter_mth' : ee.Filter.calendarRange(STARTMONTH, ENDMONTH, 'month'),
  'meta_filter_cld' : ee.Filter.lt('CLOUD_COVER', MAXCLOUD),
  'select_bands_visible' : ["SR_B1", "SR_B2","SR_B3","SR_B4"],
  'select_indices' : ["TCB", "TCG", "TCW"],
  'select_TCtrend_bands' : ["TCB_slope", "TCG_slope", "TCW_slope"],
  'geom' : None
}
#------ RUN FULL PROCESS FOR ALL REGIONS IN LOOP ------------------------------
#Map.addLayer(imageCollection, {}, 'TCVIS')

In [ ]:
RUN = 0
m = geemap.Map()

In [ ]:
for lowLat in latitudes:
    for leftLon in longitudes:
        
        
        # check for Hemisphere
        if lowLat < 0:
            sizeLat = SIZE_LAT * -1
        else:
            sizeLat = SIZE_LAT
            
        sizeLon = SIZE_LON
        
        # create Bounding Box
        geom = ee.Geometry.Polygon([leftLon,lowLat+sizeLat, leftLon, lowLat, leftLon+sizeLon, lowLat, leftLon+sizeLon, lowLat+sizeLat])
        config_trend['geom'] = geom
        m.addLayer(geom,{}, str(lowLat))

        assetname_new = utils_string.make_TCTrendAssetNameSR(leftLon, lowLat, STARTYEAR, ENDYEAR)
        assetname_nObs = assetname_new + '_nObservations'

        # Calculate Trend
        trend = high_level_functions_CO2.runTCTrend(config_trend)
        if RUN:
            task = ee.batch.Export.image.toAsset(
                image=trend['visual'],
                description=assetname_new,
                assetId=target_collection + '/' + assetname_new,
                scale=SCALE,
                region=geom,
                maxPixels=1e12)

            task.start()

            task = ee.batch.Export.image.toAsset(
                image=trend['n_observations'],
                description=assetname_nObs,
                assetId=target_collection_nObs + '/' + assetname_nObs,
                scale=SCALE,
                region=geom,
                maxPixels=1e12)

            task.start()


In [ ]:
#task.status()

In [ ]:
Map = geemap.Map()
Map.addLayer(geom)

Adding trendmap to map

In [ ]:
#Map.addLayer(ee.ImageCollection(target_collection[:-1]))

trend = high_level_functions_CO2.runTCTrend(config_trend)
Map.addLayer(trend['visual'].clip(config_trend['geom']), {
    'min': 0,
    'max': 255,
}, 'TC Trend Visual')
Map

#Map.addLayer(trend['n_observations'].clip(config_trend['geom']), {'min': 0, 'max': 50}, 'N Observations')
Map.centerObject(config_trend['geom'], 8)

Map.add_basemap(basemap='SATELLITE')
Map.addLayer(geom,{}, str(lowLat))
Map



Testing: Visualizing mosaics(doesn't work) and single images 

In [ ]:

year = 2023

annual_mosaic = trend['image_collection'].filter(ee.Filter.eq('name', year)).first()

Map.add_layer(annual_mosaic, dict(min=0.02, max=0.15, bands=['SR_B3_median','SR_B2_median','SR_B1_median']), str(year))

#Beispiel: ein Bild vom 15. Juli 2014 anzeigen
image = trend['image_collection_original'] \
   .filter(ee.Filter.date('2023-07-02', '2023-07-03')) \
   .first()

# Falls None:
image = ee.Image(image)
Map.addLayer(image, 
             {'min': 0.02, 'max': 0.15, 'bands': ['SR_B3', 'SR_B2', 'SR_B1']}, 
             'ohen L9 Bild 2023-07-02')


Testing: Images per mosaics & timestamps

In [ ]:
for year in range(2023, 2025):
    images = trend['image_collection_original'] \
        .filterDate(f'{year}-07-01', f'{year}-08-31') \
        .sort('system:time_start') \
        .toList(100)  # max 100 Bilder pro Jahr

    count = images.size().getInfo()
    print(f'{year}: {count} Bilder')

    for i in range(count):
        img = ee.Image(images.get(i))
        date = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd').getInfo()
        print(f'  - {date}')

## Export test

In [ ]:
# target_collection = 'users/ingmarnitze/TCTrend_SR_2005-2024_TCVIS'
# target_collection = 'projects/ee-ingmarnitze/assets/TCTrend_SR_2005-2024_TCVIS'
target_collection = 'projects/water-sinapohlabeln/assets/TCTrend_SR_2005-2024_TCVIS'

assetname_new = utils_string.make_TCTrendAssetNameSR(leftLon, lowLat, STARTYEAR, ENDYEAR)

assetId = target_collection + '/' + 'test_Byte'

In [ ]:
task = ee.batch.Export.image.toAsset(
    image=ee.Image(trend['visual']).toByte(),
    description=assetname_new,
    assetId=assetId,
    scale=SCALE,
    region=geom,
    maxPixels=1e12)

task.start()